In [4]:
import os
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
import numpy as np
import matplotlib as mpl
import fsspec

In [5]:
SAVE_PATH = '../scorecards'

In [6]:
CONFIG = {
    # SMALL vs ERA5
    "aurora_small_vs_era5_2021": {
        "forecast_path": "/projects/prjs1808/ewalt1/Xaurora/train/small-flow-map_600K_0.1-noise/forecast_2021_15d_20-SDE-steps/aurora-pretrained_forecasts.zarr",
        "metrics_path": "/projects/prjs1808/ewalt1/Xaurora/train/small-flow-map_600K_0.1-noise/forecast_20-SDE-steps/aurora_small_pretrained_vs_era5_1440x721_2021-01-01_2021-12-31.nc",
        "name": "Aurora small vs ERA5 2021",
        "probabilistic": False
    },
    "xaurora_small_vs_era5_2021": {
        "forecast_path": "/projects/prjs1808/ewalt1/Xaurora/train/small-flow-map_600K_0.1-noise/forecast_2021_15d_20-SDE-steps/xaurora_forecasts.zarr",
        "metrics_path": "/projects/prjs1808/ewalt1/Xaurora/train/small-flow-map_600K_0.1-noise/forecast_20-SDE-steps/xaurora_small_vs_era5_1440x721_2021-01-01_2021-12-31.nc",
        "name": "Xaurora small vs ERA5 2021",
        "probabilistic": True
    },
    "xaurora_small_fm_vs_era5_2021": {
        "forecast_path": "/projects/prjs1808/ewalt1/Xaurora/train/16g/2026-02-10_15-11-33/forecast_2026-02-16_13-05-34/xaurora_forecasts.zarr",
        "metrics_path": "/projects/prjs1808/ewalt1/Xaurora/train/16g/2026-02-10_15-11-33/forecast_2026-02-16_13-05-34/xaurora_small_fm_vs_era5_1440x721_2021-01-01_2021-12-31.nc",
        "name": "Xaurora small FM vs ERA5 2021",
        "probabilistic": True
    },
    # LARGE vs ERA5 2021
    "aurora_vs_era5_2021": {
        "forecast_path": "/projects/prjs1808/ewalt1/Xaurora/train/large-flow-map_600K_0.1-noise/forecast_2021_15d_20-SDE-steps/aurora_pretrained_forecasts.zarr",
        "metrics_path": "/projects/prjs1808/ewalt1/Xaurora/train/large-flow-map_600K_0.1-noise/forecast_2021_15d_20-SDE-steps/aurora_pretrained_vs_era5_1440x721_2021-01-01_2021-12-31.nc",
        "name": "Aurora vs ERA5 2021",
        "probabilistic": False
    },
    "xaurora_vs_era5_2021": {
        "forecast_path": "/projects/prjs1808/ewalt1/Xaurora/train/large-flow-map_600K_0.1-noise/forecast_2021_15d_20-SDE-steps/xaurora_forecasts.zarr",
        "metrics_path": "/projects/prjs1808/ewalt1/Xaurora/train/large-flow-map_600K_0.1-noise/forecast_2021_15d_20-SDE-steps/xaurora_vs_era5_1440x721_2021-01-01_2021-12-31.nc",
        "name": "Xaurora vs ERA5 2021",
        "probabilistic": True
    },
    # LARGE vs ERA5 2022
    "aurora_vs_era5_2022": {
        "forecast_path": "/projects/prjs1808/ewalt1/Xaurora/train/large-flow-map_600K_0.1-noise/forecast_2022_15d_20-SDE-steps/aurora-pretrained_forecasts.zarr",
        "metrics_path": "/projects/prjs1808/ewalt1/Xaurora/train/large-flow-map_600K_0.1-noise/forecast_2022_15d_20-SDE-steps/aurora_pretrained_vs_era5_1440x721_2022-01-01_2022-12-31.nc",
        "name": "Aurora vs ERA5 2022",
        "probabilistic": False
    },
    "xaurora_vs_era5_2022": {
        "forecast_path": "/projects/prjs1808/ewalt1/Xaurora/train/large-flow-map_600K_0.1-noise/forecast_2022_15d_20-SDE-steps/xaurora_forecasts.zarr",
        "metrics_path": "/projects/prjs1808/ewalt1/Xaurora/train/large-flow-map_600K_0.1-noise/forecast_2022_15d_20-SDE-steps/xaurora_vs_era5_1440x721_2022-01-01_2022-12-31.nc",
        "name": "Xaurora vs ERA5 2022",
        "probabilistic": True
    },
    # LARGE ZERO SHOT VS HRES T0 2022
    "aurora_finetuned_no_lora_hres_init_vs_hres_t0_2022": {
        "forecast_path": "/projects/prjs1808/ewalt1/Xaurora/train/large-flow-map_600K_0.1-noise/forecast_hres_init_2022_15d_20-SDE-steps/aurora-finetuned_forecasts.zarr",
        "metrics_path": "/projects/prjs1808/ewalt1/Xaurora/train/large-flow-map_600K_0.1-noise/forecast_hres_init_2022_15d_20-SDE-steps/aurora_finetuned_no_lora_hres_init_vs_hres_t0_1440x721_2022-01-01_2022-12-31.nc",
        "name": "Aurora finetuned no-LoRA vs HRES T0 2022",
        "probabilistic": False
    },
    "xaurora_zero_shot_hres_init_vs_hres_t0_2022": {
        "forecast_path": "/projects/prjs1808/ewalt1/Xaurora/train/large-flow-map_600K_0.1-noise/forecast_hres_init_2022_15d_20-SDE-steps/xaurora_forecasts.zarr",
        "metrics_path": "/projects/prjs1808/ewalt1/Xaurora/train/large-flow-map_600K_0.1-noise/forecast_hres_init_2022_15d_20-SDE-steps/xaurora_zero_shot_hres_init_vs_hres_t0_1440x721_2022-01-01_2022-12-31.nc",
        "name": "Xaurora zero-shot vs HRES T0 2022",
        "probabilistic": True
    },
    # LARGE FINETUNED VS HRES T0 2022
    "xaurora_hres_init_vs_hres_t0_2022": {
        "forecast_path": "/projects/prjs1808/ewalt1/Xaurora/train/large-flow-map_600K_0.1-noise_ft-hres-nowd/forecast_hres_init_2022_15d_20-SDE-steps/xaurora_forecasts.zarr",
        "metrics_path": "/projects/prjs1808/ewalt1/Xaurora/train/large-flow-map_600K_0.1-noise_ft-hres-nowd/forecast_hres_init_2022_15d_20-SDE-steps/xaurora_finetuned_hres_init_vs_hres_t0_1440x721_2022-01-01_2022-12-31.nc",
        "name": "Xaurora (Oper.) vs HRES T0 2022",
        "probabilistic": True
    },
    # GenCast operational 2022
    "gencast_hres_init_vs_era5_2022": {
        "forecast_path": None,
        "metrics_path": "/Users/eliotwalt/Documents/phd/code/weatherbenchX/scraping/scores/gencast_hres_init_vs_era5_1440x721_2022.nc",
        "name": "GenCast (Oper.) vs ERA5 2022",
        "probabilistic": True
    },
    "gencast_hres_init_vs_hres_t0_2022": {
        "forecast_path": None,
        "metrics_path": "/Users/eliotwalt/Documents/phd/code/weatherbenchX/scraping/scores/gencast_hres_init_vs_hres_t0_1440x721_2022.nc",
        "name": "GenCast (Oper.) vs HRES T0 2022",
        "probabilistic": True
    },
    # Aurora operational
    "aurora_hres_init_vs_hres_t0_2022": {
        "forecast_path": None,
        "metrics_path": "/Users/eliotwalt/Documents/phd/code/weatherbenchX/scraping/scores/aurora_hres_init_vs_hres_t0_1440x721_2022.nc",
        "name": "Aurora (Oper.) vs HRES T0 2022",
        "probabilistic": False
    },
    "custom_aurora_hres_init_vs_hres_t0_2022": {
        "forecast_path": "/projects/prjs1808/ewalt1/Xaurora/train/large-flow-map_600K_0.1-noise_ft-hres-nowd/forecast_hres_init_2022_15d_20-SDE-steps/aurora-finetuned_forecasts.zarr",
        "metrics_path": "/projects/prjs1808/ewalt1/Xaurora/train/large-flow-map_600K_0.1-noise_ft-hres-nowd/forecast_hres_init_2022_15d_20-SDE-steps/aurora_finetuned_hres_init_vs_hres_t0_1440x721_2022-01-01_2022-12-31.nc",
        "name": "Our Aurora (Oper.) vs HRES T0 2022",
        "probabilistic": False
    }
}

def scp_load(path):
    if os.path.exists(path):
        # print(f"Loading {path} from local disk")
        return xr.open_dataset(path)
    
    local_path = f'../wbx_benchmark/{os.path.basename(path)}'
    if not os.path.exists(local_path):
        # print(f"Copying snellius1:{path} to {local_path}")
        !scp -r snellius1:{path} ../wbx_benchmark/

    # rint(f"Loading {local_path} from local disk")    
    ds = xr.open_dataset(local_path)
    
    return ds

In [ ]:
plt.rcParams['figure.facecolor'] = '0.9'
# reds = sns.color_palette('Reds', 7)
# blues = sns.color_palette('Blues_r', 7)
# cmap = matplotlib.colors.ListedColormap(blues + [(0.95, 0.95, 0.95)] + reds)
# cb_levels = [-50, -25, -15, -10, -5, -2, -1, 1, 2, 5, 10, 15, 25, 50]
reds = sns.color_palette('Reds', 6)
blues = sns.color_palette('Blues_r', 6)
cmap = matplotlib.colors.ListedColormap(blues + [(0.95, 0.95, 0.95)] + reds)
cb_levels = [-50, -20,-10, -5, -2, -1, 1, 2, 5, 10, 20, 50]
norm = matplotlib.colors.BoundaryNorm(cb_levels, cmap.N, extend='both')

# define helper function
def set_y_labels(ax, dataset, levels=True):
    ax.set_xticks([])
    if levels:
      ax.set_yticks(np.arange(len(dataset.level)))
      ax.set_yticklabels(dataset.level.values)
      # ax.tick_params(axis='y', which='major', pad=10)
    else:
      ax.set_yticks([0])
      ax.tick_params(axis='y', color='None')
      ax.set_yticklabels(['000'], color='None')

    ax.spines['top'].set_color('0.7')
    ax.spines['right'].set_color('0.7')
    ax.spines['bottom'].set_color('0.7')
    ax.spines['left'].set_color('0.7')

def add_white_lines(ax, img, color='white'):
  for i in range(img.shape[0]):
    for j in range(img.shape[1]):
      rect = mpl.patches.Rectangle(
          (j - 0.5, i - 0.5),
          1,
          1,
          linewidth=2,
          edgecolor=color,
          facecolor='None',
      )
      ax.add_patch(rect)

def set_x_labels(ax, dataset, labels=None):
  lead_time_h = int(dataset.lead_time[0] / np.timedelta64(1, 'h'))
  factor_24h = 24 // lead_time_h
  xticks = np.arange(0, len(dataset.lead_time), factor_24h)
  ax.set_xticks(xticks)
  ax.set_xticklabels(xticks + 1 // factor_24h if labels is None else labels)
  ax.spines['top'].set_color('0.7')
  ax.spines['right'].set_color('0.7')
  ax.spines['bottom'].set_color('0.7')
  ax.spines['left'].set_color('0.7')

def plot_main_scorecard(relative, absolute, variables, 
                        lead_times, save_fn=None,
                        subselect_models=None, nphysical=None, nml=None,
                        custom_titles=None,
                        custom_names=None, custom_subtitles=None):
  matplotlib.rcParams.update(matplotlib.rcParamsDefault)

  models = subselect_models or absolute.model.values

  nmodels = len(models)
  nvariables = len(variables)

  panel_width = 2
  label_width = 1 * panel_width
  padding_right = 0.1

  panel_height = panel_width / 5

  title_height = panel_height * 1.25
  cbar_height = panel_height * 3
  spacing_height = panel_height * 0.1
  spacing_width = panel_height * 0.3

  total_width = label_width + nvariables * panel_width + (nvariables - 1) * spacing_width + padding_right
  total_height = title_height + cbar_height + nmodels * panel_height + (nmodels - 1) * spacing_height

  fig = plt.figure(figsize=(total_width, total_height))
  gs = mpl.gridspec.GridSpec(
      nmodels,
      nvariables,
      figure=fig,
      left=label_width / total_width,
      right=1 - padding_right / total_width,
      top=1-(title_height / total_height),
      bottom=cbar_height / total_height,
      # w/hspace are relative to average panel size
      hspace=spacing_height / panel_height,
      wspace=spacing_width / panel_width
  )
  for row, m in enumerate(models):
    for col, (var, l, metric) in enumerate(variables):
      ax = fig.add_subplot(gs[row, col])

      abs = absolute.sel(model=m, metric=metric)[var]
      da = relative.sel(model=m, metric=metric)[var]
      if l:
        da = da.sel(level=l)
        abs = abs.sel(level=l)
      values = da.sel(lead_time=lead_times).values[None, :]
      abs_values = abs.sel(lead_time=lead_times).values[None, :]
      img = ax.imshow(values, aspect='auto', cmap=cmap, norm=norm)
      set_y_labels(ax, da, levels=False)
      add_white_lines(ax, values, color='0.9')
      ax.grid(False)
      if col == 0:
        if custom_names is None:
          label = m.split(' vs')[0]
        else:
          label = custom_names[row]
        ax.set_ylabel(label, rotation='horizontal', horizontalalignment='right',
                      verticalalignment='center', labelpad=-20, zorder=10, fontsize=9.5)
        # Add physical model box
        if (nphysical is not None) and (row == nphysical-1):
          h=nphysical + (nphysical - 1) * spacing_height / panel_height
          rect = mpl.patches.Rectangle(
              (-0.98, 0), width=0.96, height=h, color="lightsteelblue", transform=ax.transAxes,
              clip_on=False, alpha=0.2, zorder=0.1
          )
          ax.add_patch(rect)
          if nphysical >=3:
            ax.text(-0.9, h/2, 'Physical models', ha='center',va='center',
                    transform=ax.transAxes, fontsize=8, rotation='vertical')
        # Add ML model box
        elif (nml is not None) and (row == nphysical + nml - 1):
          h=nml + (nml - 1) * spacing_height / panel_height
          rect = mpl.patches.Rectangle(
              (-0.98, 0), width=0.96, height=h, color="lightcoral", transform=ax.transAxes,
              clip_on=False, alpha=0.2, zorder=0.1
          )
          ax.add_patch(rect)
          if nml >=3:
            ax.text(-0.9, h/2, 'ML / hybrid models', ha='center',va='center',
                    transform=ax.transAxes, fontsize=8, rotation='vertical')
        else:
          ax.set_zorder(10)
      if row == 0:

        ax.set_title(custom_titles[col], fontsize=10, pad=17)
        ax.text(0.5, 1.25, custom_subtitles[col], ha='center',va='center', transform=ax.transAxes,
                fontsize=6)
      if np.isnan(values).all():
        ax.remove()
        continue


      if row == nmodels - 1:
        lead_time_in_days = absolute.lead_time.values.astype('timedelta64[h]').astype(int) // 24
        set_x_labels(ax, da, labels=lead_time_in_days)
        ax.set_xlabel('Lead time [days]')
      # Add absolute numbers
      for i, v in enumerate(abs_values[0]):
        if m == 'Climatology vs ERA5':
          v = np.mean(abs_values)
        if np.isfinite(v):
          if var == 'Specific Humidity': v *= 1000
          if var == '24h Precipitation' and metric == 'CRPS': v *= 1000

          if v > 10:
            v = str(v)[:3].rstrip('.')
          else:
            v = str(v)[:4]
          ax.text(i, 0, v, ha='center',va='center', fontsize=8)

  rel_cbar_height = cbar_height / total_height
  cax = fig.add_axes((0.35, rel_cbar_height * 0.4, 0.5, rel_cbar_height * 0.1))
  cb = fig.colorbar(img, cax=cax, orientation='horizontal', fraction=0.01)
  cb.ax.set_xticks(cb_levels)
  if '24h Precipitation' in np.ravel(variables):
    cb.ax.set_xlabel(r'Better $\longleftarrow$ % difference in RMSE/SEEPS vs IFS HRES $\longrightarrow$ Worse')
  else:
    cb.ax.set_xlabel(r'Better $\longleftarrow$ % difference in RMSE vs IFS HRES $\longrightarrow$ Worse')

  if save_fn:
    with fsspec.open(f'{SAVE_PATH}/{save_fn}', 'wb', auto_mkdir=True) as f:
      # fig.patch.set_facecolor('gray')
      plt.savefig(f, dpi=300)

      plt.close()

In [ ]:
lead_times = np.array([1, 3, 5, 7, 10, 15], dtype='timedelta64[D]')